# CS 577 Homework 2: Inside an MLP

**Student starter notebook**

This notebook follows one model all the way through:

`affine -> ReLU -> affine -> BCE-with-logits -> manual backward -> gradient checks -> XOR experiment`

Restart the kernel and run every cell before saving. Use `float64` for all required gradient comparisons.

In [2]:
import platform
import subprocess
import sys

In [3]:
    import platform
    import subprocess
    import sys

    import matplotlib.pyplot as plt
    import numpy as np
    import torch
    import torch.nn.functional as F

    np.set_printoptions(precision=6, suppress=True)
    print("Python:", sys.version.split()[0])
    print("Platform:", platform.platform())
    print("NumPy:", np.__version__)
    print("PyTorch:", torch.__version__)
    print("Required device: cpu")

Python: 3.13.5
Platform: macOS-15.7.4-arm64-arm-64bit-Mach-O
NumPy: 2.5.2
PyTorch: 2.13.0
Required device: cpu


## 0. Environment, public tests, and shape contract

Run the tests after implementing `hw2_mlp.py`. In the written report, explain `(B,1)` versus `(B,)` and where the batch-mean factor enters backward propagation.

In [5]:
result = subprocess.run(
    [sys.executable, "-m", "unittest", "-v", "test_hw2_public.py"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)


test_08_prediction_helpers_and_accuracy_shape_guard (test_hw2_public.ExperimentHelperTests.test_08_prediction_helpers_and_accuracy_shape_guard) ... ERROR
test_09_xor_data_shape_balance_and_reproducibility (test_hw2_public.ExperimentHelperTests.test_09_xor_data_shape_balance_and_reproducibility) ... ok
test_10_training_returns_history_updates_and_learns (test_hw2_public.ExperimentHelperTests.test_10_training_returns_history_updates_and_learns) ... ERROR
test_04_initialization_shapes_dtype_and_determinism (test_hw2_public.ForwardBackwardTests.test_04_initialization_shapes_dtype_and_determinism) ... ok
test_05_forward_values_cache_and_no_mutation (test_hw2_public.ForwardBackwardTests.test_05_forward_values_cache_and_no_mutation) ... ERROR
test_06_manual_backward_values_shapes_and_no_update (test_hw2_public.ForwardBackwardTests.test_06_manual_backward_values_shapes_and_no_update) ... ERROR
test_07_finite_differences_match_manual_and_do_not_mutate (test_hw2_public.ForwardBackwardTests.test

In [ ]:
import hw2_mlp as mlp

## 1-2. Fixed-case forward and manual backward

In [6]:
X_check = np.array([
    [ 0.2, -0.4],
    [ 1.0,  0.5],
    [-0.3,  0.8],
    [ 0.7, -1.2],
], dtype=np.float64)

y_check = np.array([[0.0], [1.0], [1.0], [0.0]], dtype=np.float64)

params_check = {
    "W1": np.array([[ 0.5, -0.3,  0.8],
                    [-0.7,  0.6,  0.2]], dtype=np.float64),
    "b1": np.array([0.1, -0.2, 0.3], dtype=np.float64),
    "W2": np.array([[ 0.4], [-0.5], [0.7]], dtype=np.float64),
    "b2": np.array([-0.1], dtype=np.float64),
}

z1_check = X_check @ params_check["W1"] + params_check["b1"]
print("minimum |Z1| in check case:", np.min(np.abs(z1_check)))

minimum |Z1| in check case: 0.2


In [8]:
logits_check, cache_check = mlp.mlp_forward(X_check, params_check)
print("cache shapes:", {key: value.shape for key, value in cache_check.items()})
print("logits:\n", logits_check)

loss_check, manual_grads = mlp.mlp_loss_and_gradients(X_check, y_check, params_check)
print("loss:", loss_check)
for key in mlp.PARAMETER_KEYS:
    print(key, "shape=", manual_grads[key].shape, "norm=", np.linalg.norm(manual_grads[key]))

NameError: name 'mlp' is not defined

## 3. Centered finite differences

In [ ]:
numerical_grads = mlp.finite_difference_gradients(
    X_check, y_check, params_check, epsilon=1e-5
)
for key in mlp.PARAMETER_KEYS:
    max_abs = np.max(np.abs(manual_grads[key] - numerical_grads[key]))
    rel = relative_error(manual_grads[key], numerical_grads[key])
    print(f"{key}: max_abs={max_abs:.3e}, relative={rel:.3e}")

## 3.2 PyTorch autograd comparison

Recreate the fixed model using independent `torch.float64` tensors with `requires_grad=True`. Compute `binary_cross_entropy_with_logits`, call `backward()`, and build a dictionary named `torch_grads` with NumPy arrays for `W1`, `b1`, `W2`, and `b2`.

In [ ]:
# TODO: implement the independent PyTorch reference.
torch_grads = {}

# Then report errors, for example:
# for key in mlp.PARAMETER_KEYS:
#     max_abs = np.max(np.abs(manual_grads[key] - torch_grads[key]))
#     print(key, max_abs, relative_error(manual_grads[key], torch_grads[key]))

## 4. Nonlinear XOR experiment

In [ ]:
def relative_error(a, b):
    numerator = np.linalg.norm(a - b)
    denominator = np.linalg.norm(a) + np.linalg.norm(b) + 1e-12
    return float(numerator / denominator)


def split_data(X, y, train_fraction=0.75, seed=2026):
    rng = np.random.default_rng(seed)
    order = rng.permutation(X.shape[0])
    cut = int(train_fraction * X.shape[0])
    train_idx, test_idx = order[:cut], order[cut:]
    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]


def train_linear_logistic(X, y, steps=2000, lr=0.2):
    # Provided full-batch linear baseline: one affine score plus sigmoid.
    w = np.zeros((X.shape[1], 1), dtype=np.float64)
    b = np.zeros(1, dtype=np.float64)
    history = []
    for _ in range(steps):
        logits = X @ w + b
        loss = mlp.binary_cross_entropy_with_logits(logits, y)
        history.append(loss)
        dlogits = (mlp.sigmoid_stable(logits) - y) / X.shape[0]
        w -= lr * (X.T @ dlogits)
        b -= lr * np.sum(dlogits, axis=0)
    return {"w": w, "b": b}, history


def linear_probability(X, linear_params):
    return mlp.sigmoid_stable(X @ linear_params["w"] + linear_params["b"])


def metrics_from_probabilities(probability, y):
    eps = 1e-12
    probability = np.clip(probability, eps, 1.0 - eps)
    loss = -np.mean(y * np.log(probability) + (1.0 - y) * np.log(1.0 - probability))
    accuracy = np.mean((probability >= 0.5) == y)
    return float(loss), float(accuracy)


def plot_boundary(ax, probability_function, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 240),
        np.linspace(y_min, y_max, 240),
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = probability_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.65)
    ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="bwr", edgecolors="black", s=28)
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

In [ ]:
X_all, y_all = mlp.make_xor_data(n_per_quadrant=60, noise=0.28, seed=577)
X_train, y_train, X_test, y_test = split_data(X_all, y_all, train_fraction=0.75, seed=2026)
print("train:", X_train.shape, y_train.shape, "test:", X_test.shape, y_test.shape)

linear_params, linear_history = train_linear_logistic(X_train, y_train)
linear_train = metrics_from_probabilities(linear_probability(X_train, linear_params), y_train)
linear_test = metrics_from_probabilities(linear_probability(X_test, linear_params), y_test)
print("linear train loss/accuracy:", linear_train)
print("linear test  loss/accuracy:", linear_test)

In [ ]:
mlp_params, mlp_history = mlp.train_mlp(
    X_train, y_train, hidden_dim=8, steps=2000, lr=0.2, seed=577
)
mlp_train = metrics_from_probabilities(mlp.predict_proba(X_train, mlp_params), y_train)
mlp_test = metrics_from_probabilities(mlp.predict_proba(X_test, mlp_params), y_test)
print("MLP train loss/accuracy:", mlp_train)
print("MLP test  loss/accuracy:", mlp_test)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
plot_boundary(
    axes[0],
    lambda grid: linear_probability(grid, linear_params),
    X_all,
    y_all,
    "Linear logistic baseline",
)
plot_boundary(
    axes[1],
    lambda grid: mlp.predict_proba(grid, mlp_params),
    X_all,
    y_all,
    "Two-layer ReLU MLP",
)
axes[2].plot(linear_history, label="linear")
axes[2].plot(mlp_history, label="MLP")
axes[2].set_yscale("log")
axes[2].set_xlabel("step")
axes[2].set_ylabel("training BCE")
axes[2].set_title("Training loss")
axes[2].legend()
fig.tight_layout()
fig.savefig("hw2_experiment.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Written interpretation and reflection checklist

Move the requested derivations, comparison tables, experiment interpretation, reflections, and assistance/AI disclosure into your written PDF. Make sure the notebook retains the numerical evidence and plots used by that report.